# 🔎 burst_scan — do the detectors find real moves? (gate G0)

Before any model is trusted, each sub-strategy's **detector alone** must pass: after the events it flags, does
price reach a target *in that side's direction* before an equal stop **more often than after comparable ordinary
bars**? Everything here runs on the training window only (to 2026-04-30); no classifier is involved.

| § | question |
|---|---|
| 1 | load the 1-minute bars; derive 5- and 10-minute bars from them |
| 2 | **anatomy** — how many events, when in the day, how clustered, how often both sides fire together |
| 3 | **the controls** — how the two placebos are built, and whether they really match the events |
| 4 | **the outcome grid** — win / stop / time-out rates and the demeaned return over 19 hold × bar cells, events against both placebos |
| 5 | **dose-response** — does the win rate rise with the size of the burst? and how much of the return is overnight? |
| 6 | **the detector grid** — every pre-registered detector setting per side; the pick rule applied, the values printed for `config.yaml` |
| 7 | drift by feature quintile (research only — in-sample edges) |
| 8 | **G0 verdict** per side, checks and the scorecard |

**The two placebos.** *A*: for every event, a random bar of the **same name, month and half hour** with no burst
within `cooldown` either side — "an ordinary moment of this stock". *B*: a bar with a **same-size price move**
(same sign, same size decile, same month) but **normal volume** — "a move without the crowd". If events beat A
but not B, the detector is just a big-move detector and volume adds nothing.

**G0, pre-registered (README).** At the primary cell (5-minute bars, one-session hold) and the grid's pick, a side
passes when: its win rate ≥ 1.25 × placebo A's with a day-clustered t ≥ 3; it beats placebo B with t ≥ 2; wins
beat stops on the same events (t ≥ 2); and its mean market-demeaned return is positive. A side that fails stops
here — no model is fit on it.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import plots, strategies
from trading_simple.config import load_settings
from trading_simple.data import resample_panel
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


SUBSET = None  # e.g. 40 for a quick first run; None = all 200 names
OVERRIDES = {}  # any signals() knob
PRM = {**strat.params, **OVERRIDES}
SEED = 0
BARS = ("1m", "5m", "10m")
P_BAR, P_H = rs.PRIMARY  # the pre-registered primary cell: 5-minute bars, a one-session hold

## 1 · Load the minutes once

The 1-minute panel is loaded once; 5- and 10-minute candles are derived from it (`data.resample_panel`), and
**every quantity is recomputed on each bar size's own grid** — a 1-minute feature is never joined onto a
5-minute event. The contexts are built one bar size at a time and reduced to tables, so only one lives in memory.

In [ ]:
panel1 = rs.load(settings, engine, subset=SUBSET, resample=False)
print(
    f"1m panel: {panel1.close.shape[0]:,} bars x {len(panel1.symbols)} names "
    f"({panel1.close.index[0]:%Y-%m-%d} → {panel1.close.index[-1]:%Y-%m-%d})"
)


def at_bar(bar):
    return panel1 if bar == "1m" else resample_panel(panel1, int(bar[:-1]))

### 1b · Score every cell once

For every bar size and hold of the pre-registered grid, each side's events and their two placebos are scored the
way the engine would book them (target/stop at `move_atr` session-ATRs × √(hold / 390) from the fill bar's close,
checked from the next bar, a bar touching both is a stop, a gap fills at the open; "close" = flat at the session's
last bar). The **paired** columns compare each event with *its own* placebo, day-clustered. §2–§8 only read what
this cell computes (the primary bar's context is kept for them).

In [ ]:
rows_out, lifts, keep = [], [], {}
for bar in BARS:
    pb = at_bar(bar)
    ctx = S["context"](pb, **PRM)
    pool = rs.placebo_pool(ctx, cooldown_bars=S["_bars"](PRM["cooldown"], bar))
    for side in SIDES:
        q = S["_side_params"](PRM, side)
        r, c_ = rs.events(ctx, PRM, side)
        ar, ac, ma = rs.draw_placebo_a(pool, ctx, r, c_, seed=SEED)
        br, bc, mb = rs.draw_placebo_b(ctx, r, c_, seed=SEED)
        if bar == P_BAR:
            keep[side] = {"rows": r, "cols": c_, "a": (ar, ac, ma), "b": (br, bc, mb)}
        for h in rs.CELLS[bar]:
            kw = {"side": side, "horizon": h, "move_atr": q["move_atr"]}
            oc_e = rs.side_outcomes(pb, ctx, r, c_, **kw)
            oc_a = rs.side_outcomes(pb, ctx, ar, ac, **kw)
            oc_b = rs.side_outcomes(pb, ctx, br, bc, **kw)
            hold = h if h else "close"
            for name, oc in (("events", oc_e), ("placebo A", oc_a), ("placebo B", oc_b)):
                rows_out.append(
                    {"bar": bar, "hold": hold, "side": side, "set": name, **rs.summarize(oc)}
                )
            la = rs.lift(oc_e[ma].reset_index(drop=True), oc_a)
            lb = rs.lift(oc_e[mb].reset_index(drop=True), oc_b)
            wl = rs.day_t(oc_e["win"].astype(float) - oc_e["reason"].eq(-1), oc_e["day"])
            lifts.append(
                {
                    "bar": bar,
                    "hold": hold,
                    "side": side,
                    "lift A": la["lift"],
                    "t A": la["t"],
                    "lift B": lb["lift"],
                    "t B": lb["t"],
                    "matched B": mb.mean(),
                    "win − stop": wl["mean"],
                    "t win−stop": wl["t"],
                }
            )
    if bar == P_BAR:
        pb5, ctx5, pool5 = pb, ctx, pool
    else:
        del ctx
grid_out = pd.DataFrame(rows_out)
lift_out = pd.DataFrame(lifts)
print(f"{len(grid_out)} outcome rows")

## 2 · Anatomy (primary bar)

How often each detector fires, where in the day, and how clustered the raw bursts are: `quiet_first` keeps only
the first burst after `cooldown` quiet minutes, so a news day that stays busy counts once per quiet spell.
`both sides` counts bars where the long and the short detector fire on the same name.

In [ ]:
mpb = S["_minutes_per_bar"](P_BAR)
n_names, n_sess = len(pb5.symbols), len(np.unique(ctx5["codes"][ctx5["window"].any(axis=1)]))
anat = {}
for side in SIDES:
    q = S["_side_params"](PRM, side)
    r, c_ = keep[side]["rows"], keep[side]["cols"]
    raw = S["detect"](
        ctx5,
        trigger=q["trigger"],
        impulse=q["impulse"],
        vol_mult=q["vol_mult"],
        range_mult=q["range_mult"],
        cooldown=0,
    )
    anat[side] = {
        "events": len(r),
        "per name per session": len(r) / (n_names * n_sess),
        "raw bursts in window": int(raw.sum()),
        "kept by quiet_first": len(r) / max(int(raw.sum()), 1),
        "up bursts": float((ctx5["move"][r, c_] > 0).mean()),
        "busiest name share": pd.Series(c_).value_counts(normalize=True).iloc[0],
    }
both = len(
    set(zip(keep["long"]["rows"], keep["long"]["cols"], strict=True))
    & set(zip(keep["short"]["rows"], keep["short"]["cols"], strict=True))
)
display(pd.DataFrame(anat).round(3))
print(f"both sides on the same bar and name: {both:,}")

fig = make_subplots(rows=1, cols=2, subplot_titles=("events by time of day", "events per session"))
for side in SIDES:
    r = keep[side]["rows"]
    tod = ctx5["feat"]["tod"][r]
    fig.add_trace(
        go.Histogram(x=tod, name=side, marker_color=COLOR[side], opacity=0.6, xbins={"size": 15}),
        row=1,
        col=1,
    )
    per = (
        pd.Series(ctx5["codes"][r])
        .value_counts()
        .reindex(np.unique(ctx5["codes"][ctx5["window"].any(axis=1)]), fill_value=0)
    )
    fig.add_trace(
        go.Histogram(x=per, name=side, marker_color=COLOR[side], opacity=0.6, showlegend=False),
        row=1,
        col=2,
    )
fig.update_layout(barmode="overlay")
plots._base(
    fig,
    title="Fig 2 · when the detectors fire (minutes since the open; events per session)",
    height=400,
)
below(fig, -0.15)
fig.show()

## 3 · Do the placebos match?

Placebo A matches on name, month and half hour **by construction** — the time-of-day histograms below should lie on
top of each other. Placebo B matches on month, the move's sign and its size decile; its match rate says how often a
same-size move happens *without* a crowd — a low rate is itself a finding (big moves on normal volume are rare).

In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=[f"{s}: time of day, events vs placebo A" for s in SIDES]
)
match = {}
for i, side in enumerate(SIDES, start=1):
    k_ = keep[side]
    ar, ac, ma = k_["a"]
    br, bc, mb = k_["b"]
    fig.add_trace(
        go.Histogram(
            x=ctx5["feat"]["tod"][k_["rows"][ma]],
            name="events",
            marker_color=COLOR[side],
            opacity=0.55,
            xbins={"size": 15},
            showlegend=i == 1,
        ),
        row=1,
        col=i,
    )
    fig.add_trace(
        go.Histogram(
            x=ctx5["feat"]["tod"][ar],
            name="placebo A",
            marker_color=plots._MUTED,
            opacity=0.55,
            xbins={"size": 15},
            showlegend=i == 1,
        ),
        row=1,
        col=i,
    )
    size = np.abs(ctx5["feat"]["burst_atr"])
    match[side] = {
        "A matched": ma.mean(),
        "A same name": bool((ac == k_["cols"][ma]).all()),
        "B matched": mb.mean(),
        "B median |move| ATR (events)": float(np.nanmedian(size[k_["rows"][mb], k_["cols"][mb]])),
        "B median |move| ATR (placebo)": float(np.nanmedian(size[br, bc])),
        "B median rel. volume (placebo)": float(np.nanmedian(ctx5["vol_ratio"][br, bc])),
    }
fig.update_layout(barmode="overlay")
plots._base(fig, title="Fig 3 · placebo A sits at the same clock times as the events", height=380)
below(fig, -0.15)
fig.show()
pd.DataFrame(match).round(3)

## 4 · The outcome grid

**Win rate** = target reached first (the side's own direction). Every column is at the side's own detector as
configured; §6 re-picks the detector. A real detector shows events ABOVE both placebos in the same cell — a cell
where events and placebo A are equal says the detector flags ordinary moments. `lift A` / `lift B` are the win
rate over the matched placebo's, `t` day-clustered on the event-by-event difference.

In [ ]:
wide = grid_out.pivot_table(
    index=["bar", "hold"], columns=["side", "set"], values="win", sort=False
)
display(wide.round(3))
display(lift_out.set_index(["bar", "hold", "side"]).round(2))

fig = make_subplots(
    rows=2,
    cols=3,
    shared_yaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[f"{s} · {b}" for s in SIDES for b in BARS],
)
DASH = {"events": "solid", "placebo A": "dash", "placebo B": "dot"}
for i, side in enumerate(SIDES, start=1):
    for j, bar in enumerate(BARS, start=1):
        g = grid_out[(grid_out["side"] == side) & (grid_out["bar"] == bar)]
        for name in ("events", "placebo A", "placebo B"):
            s_ = g[g["set"] == name]
            fig.add_trace(
                go.Scatter(
                    x=s_["hold"].astype(str),
                    y=s_["win"],
                    name=name,
                    line={
                        "color": COLOR[side] if name == "events" else plots._MUTED,
                        "dash": DASH[name],
                    },
                    showlegend=(i, j) == (1, 1),
                ),
                row=i,
                col=j,
            )
plots._base(
    fig,
    title="Fig 4 · win rate by hold: events (solid) against placebo A (dash) and B (dot)",
    height=640,
)
below(fig, -0.08)
fig.update_yaxes(tickformat=".0%")
fig.show()

dr = grid_out.pivot_table(
    index=["bar", "hold"], columns=["side", "set"], values="dret_bp", sort=False
)
print("mean market-demeaned return, bp (signed as positioned; costs NOT deducted)")
dr.round(1)

## 5 · Dose-response and the overnight share

A detector that measures something real should get **more reliable as the burst gets bigger**. Each side's
detector is re-run at a ladder of thresholds — volume alone, then range alone — and at each rung the rate of
**wins** (target first, solid) and of **stops** (dotted) is set against placebo A's win rate (dashed). Watch the
GAP between the solid and the dotted line: if both rise together, bigger bursts only mean bigger moves *both
ways* (volatility, which a barrier trade cannot use); if wins pull away from stops, the burst carries direction.
The table below it splits the primary cell's return into the part earned **overnight** (the gaps the position
held) and the part earned **during sessions** — these names earned their money overnight, so an "edge" that is
all overnight gap is exposure, not detection.

In [ ]:
qp = {s: S["_side_params"](PRM, s) for s in SIDES}
LADDER = {"volume": (1.5, 2.0, 3.0, 4.0, 6.0, 8.0), "range": (1.5, 2.0, 3.0, 4.0, 6.0)}
dose = []
for side in SIDES:
    kw = {"side": side, "horizon": P_H, "move_atr": qp[side]["move_atr"]}
    for trig, ladder in LADDER.items():
        for level in ladder:
            knob = {"vol_mult": level} if trig == "volume" else {"range_mult": level}
            r, c_ = rs.events(ctx5, PRM, side, trigger=trig, impulse="any", **knob)
            if len(r) < 30:
                continue
            ev_a, pl_a = rs.matched_outcomes(
                pb5, ctx5, r, c_, lambda a, b: rs.draw_placebo_a(pool5, ctx5, a, b, seed=SEED), **kw
            )
            ok = (ev_a["complete"] & pl_a["complete"]).to_numpy()
            dose.append(
                {
                    "side": side,
                    "trigger": trig,
                    "threshold": level,
                    "events": len(r),
                    "win": ev_a["win"][ok].mean(),
                    "stop": ev_a["reason"][ok].eq(-1).mean(),
                    "placebo win": pl_a["win"][ok].mean(),
                    "dret bp": 1e4 * ev_a["dret"][ok].mean(),
                }
            )
dose = pd.DataFrame(dose)
fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=(
        "volume threshold (x its clock-time norm)",
        "range threshold (x its clock-time norm)",
    ),
)
for j, trig in enumerate(LADDER, start=1):
    for side in SIDES:
        d = dose[(dose["side"] == side) & (dose["trigger"] == trig)]
        for col_, dash, name in (
            ("win", "solid", "win"),
            ("stop", "dot", "stop"),
            ("placebo win", "dash", "placebo A win"),
        ):
            fig.add_trace(
                go.Scatter(
                    x=d["threshold"],
                    y=d[col_],
                    mode="lines+markers",
                    name=f"{side}: {name}",
                    showlegend=j == 1,
                    line={"color": COLOR[side], "dash": dash, "width": 2 if col_ == "win" else 1},
                ),
                row=1,
                col=j,
            )
plots._base(fig, title=f"Fig 5 · reliability by burst size ({P_BAR}, hold {P_H} min)", height=420)
below(fig, -0.18)
fig.update_xaxes(type="log")
fig.update_yaxes(tickformat=".0%")
fig.show()
display(dose.round(3))

night = {}
for side in SIDES:
    er, ec = keep[side]["rows"], keep[side]["cols"]
    oc_e = rs.side_outcomes(
        pb5, ctx5, er, ec, side=side, horizon=P_H, move_atr=qp[side]["move_atr"]
    )
    tab = oc_e.assign(row=er, col=ec)[oc_e["complete"].to_numpy()]
    parts = rs.split_overnight(pb5, ctx5, tab, side)
    night[side] = {
        "total bp": 1e4 * parts["total"].mean(),
        "overnight bp": 1e4 * parts["overnight"].mean(),
        "intraday bp": 1e4 * parts["intraday"].mean(),
        "held overnight": tab["nights"].gt(0).mean(),
    }
print("primary cell, events as configured: signed log return split (bp, before costs)")
pd.DataFrame(night).round(2)

## 6 · The detector grid and its pick (per side)

Every pre-registered detector setting (`research.GRID`: trigger × impulse × volume threshold × range threshold,
deduplicated) at the primary cell, each against its own placebo A draw. **The pick rule, fixed in advance:** the
highest `lift` among settings with at least one event per name per five sessions (`eligible`). Nothing is written
to `config.yaml` automatically — the values to set are printed, and the detector is frozen from then on (never
refit on model or backtest results).

In [ ]:
grids, picks = {}, {}
for side in SIDES:
    g = rs.detector_grid(
        pb5, ctx5, PRM, side, pool5, horizon=P_H, move_atr=qp[side]["move_atr"], seed=SEED
    )
    grids[side], picks[side] = g, rs.pick_detector(g)
    print(f"\n{side}: {len(g)} settings, {int(g['eligible'].sum())} eligible")
    display(g.sort_values("lift", ascending=False).head(10).round(3))
for side, pk in picks.items():
    if pk is None:
        print(f"{side}: no eligible setting — nothing to pick")
        continue
    vm = pk["vol_mult"] if np.isfinite(pk["vol_mult"]) else qp[side]["vol_mult"]
    rm = pk["range_mult"] if np.isfinite(pk["range_mult"]) else qp[side]["range_mult"]
    print(f"{side} pick → config.yaml (lift {pk['lift']:.2f}, t {pk['t']:.1f}):")
    for knob, val in (
        ("trigger", pk["trigger"]),
        ("impulse", pk["impulse"]),
        ("vol_mult", f"{vm:g}"),
        ("range_mult", f"{rm:g}"),
    ):
        print(f"  {side}_{knob}: {val}")

## 7 · Drift by feature quintile (research only)

For each context feature, the primary cell's win rate by quintile of that feature, per side. This is where a
classifier could find something — or where it will find nothing. **In-sample**: the quintile edges and the
reading come from the same events, so nothing here is a result; `burst_scan_model.ipynb` is where features are
tested out of sample.

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[f"{s}: win rate − overall, by quintile" for s in SIDES],
    horizontal_spacing=0.18,
)
for i, side in enumerate(SIDES, start=1):
    er, ec = keep[side]["rows"], keep[side]["cols"]
    oc = rs.side_outcomes(pb5, ctx5, er, ec, side=side, horizon=P_H, move_atr=qp[side]["move_atr"])
    feats = S["features"](ctx5, er, ec)[oc["complete"].to_numpy()]
    win = oc.loc[oc["complete"], "win"].to_numpy(float)
    heat = {}
    for f in S["FEATURES"]:
        v = feats[f].to_numpy(float)
        ok = np.isfinite(v)
        if ok.sum() < 50 or np.unique(v[ok]).size < 5:
            continue
        qs = pd.qcut(v[ok], 5, labels=False, duplicates="drop")
        heat[f] = pd.Series(win[ok]).groupby(qs).mean() - win[ok].mean()
    hm = pd.DataFrame(heat).T
    fig.add_trace(
        go.Heatmap(
            z=hm.to_numpy(),
            x=[f"Q{c + 1}" for c in hm.columns],
            y=hm.index,
            colorscale=plots.DIVERGING,
            zmid=0,
            zmin=-0.08,
            zmax=0.08,
            showscale=i == 2,
        ),
        row=1,
        col=i,
    )
plots._base(
    fig,
    title="Fig 7 · win rate minus the side's average, by feature quintile (in-sample)",
    height=560,
)
fig.show()

## 8 · G0 verdict (per side)

Measured at the primary cell on the §6 pick. The config's own detector is shown alongside — if it differs from the
pick, set the pick in `config.yaml` (§6 prints the lines) and re-run this notebook before fitting any model.

In [ ]:
verdict, checks = {}, []
for side in SIDES:
    for label, det in (
        ("config", {}),
        (
            "grid pick",
            {}
            if picks[side] is None
            else {
                "trigger": picks[side]["trigger"],
                "impulse": picks[side]["impulse"],
                **(
                    {"vol_mult": picks[side]["vol_mult"]}
                    if np.isfinite(picks[side]["vol_mult"])
                    else {}
                ),
                **(
                    {"range_mult": picks[side]["range_mult"]}
                    if np.isfinite(picks[side]["range_mult"])
                    else {}
                ),
            },
        ),
    ):
        r, c_ = rs.events(ctx5, PRM, side, **det)
        kw = {"side": side, "horizon": P_H, "move_atr": qp[side]["move_atr"]}
        ev_a, pl_a = rs.matched_outcomes(
            pb5, ctx5, r, c_, lambda a, b: rs.draw_placebo_a(pool5, ctx5, a, b, seed=SEED), **kw
        )
        ev_b, pl_b = rs.matched_outcomes(
            pb5, ctx5, r, c_, lambda a, b: rs.draw_placebo_b(ctx5, a, b, seed=SEED), **kw
        )
        oc = rs.side_outcomes(pb5, ctx5, r, c_, **kw)
        la, lb = rs.lift(ev_a, pl_a), rs.lift(ev_b, pl_b)
        ok = oc["complete"]
        wl = rs.day_t(
            oc.loc[ok, "win"].astype(float) - oc.loc[ok, "reason"].eq(-1), oc.loc[ok, "day"]
        )
        dret = rs.day_t(oc.loc[ok, "dret"], oc.loc[ok, "day"])
        gates = {
            "lift A ≥ 1.25, t ≥ 3": bool(la["lift"] >= 1.25 and la["t"] >= 3),
            "beats placebo B, t ≥ 2": bool(la["win"] > lb["placebo_win"] and lb["t"] >= 2),
            "wins beat stops, t ≥ 2": bool(wl["mean"] > 0 and wl["t"] >= 2),
            "demeaned return > 0": bool(dret["mean"] > 0),
        }
        verdict[(side, label)] = {
            "events": len(r),
            "win": la["win"],
            "placebo A": la["placebo_win"],
            "lift A": la["lift"],
            "t A": la["t"],
            "placebo B": lb["placebo_win"],
            "t B": lb["t"],
            "stop rate": oc.loc[ok, "reason"].eq(-1).mean(),
            "t win−stop": wl["t"],
            "dret bp": 1e4 * dret["mean"],
            "t dret": dret["t"],
            **gates,
            "G0": "PASS" if all(gates.values()) else "FAIL",
        }
v = pd.DataFrame(verdict).T
display(v.round(3))
for side in SIDES:
    row = v.loc[(side, "grid pick")]
    checks.append((f"{side}: G0 at the grid pick", row["G0"], "PASS", row["G0"] == "PASS"))
checks.append(
    (
        "placebo A: same name for every matched event",
        all(match[s]["A same name"] for s in SIDES),
        True,
        all(match[s]["A same name"] for s in SIDES),
    )
)
checks.append(
    (
        "placebo A match rate",
        f"{min(match[s]['A matched'] for s in SIDES):.1%}",
        "≥ 90%",
        min(match[s]["A matched"] for s in SIDES) >= 0.9,
    )
)
checks.append(
    (
        "placebo B match rate (reported, low = moves without a crowd are rare)",
        f"{min(match[s]['B matched'] for s in SIDES):.1%}",
        "> 0",
        min(match[s]["B matched"] for s in SIDES) > 0,
    )
)
display(rs.checks_table(checks))
rs.scorecard(
    {
        "notebook": "events",
        "study": strat.optuna["study"],
        "names": len(pb5.symbols),
        "primary": {"bar": P_BAR, "hold": P_H},
        "G0": {
            f"{s}|{lab}": {
                k: verdict[(s, lab)][k]
                for k in (
                    "G0",
                    "events",
                    "win",
                    "placebo A",
                    "lift A",
                    "t A",
                    "t B",
                    "t win−stop",
                    "dret bp",
                )
            }
            for (s, lab) in verdict
        },
        "picks": {
            s: (
                None
                if p is None
                else {
                    k: p[k] for k in ("trigger", "impulse", "vol_mult", "range_mult", "lift", "t")
                }
            )
            for s, p in picks.items()
        },
    }
)